# 01-02 Tensor 形状：索引、形状变换、维度交换、拼接与广播

这一份只讲怎么取数据、怎么改 shape、怎么交换维度、怎么拼接张量，以及广播机制。


In [ ]:
import torch
import numpy as np

torch.manual_seed(42)
print("torch version:", torch.__version__)


## 8. 索引和切片：拿出你想要的部分

索引规则和 NumPy 很像。

| 写法 | 说明 |
|---|---|
| `x[0]` | 第 0 行 |
| `x[:, 0]` | 所有行，第 0 列 |
| `x[1:3]` | 第 1 到第 2 行，左闭右开 |
| `x[:2, 1:]` | 前两行，从第 1 列到最后 |
| `x[x > 0]` | 布尔索引，取满足条件的元素 |

注意：Python 从 0 开始计数。`1:3` 包含 1，不包含 3。

In [ ]:
x = torch.arange(12).reshape(3, 4)

print("x =\n", x)
print("x[0] =", x[0])
print("x[:, 0] =", x[:, 0])
print("x[1:3] =\n", x[1:3])
print("x[:2, 1:] =\n", x[:2, 1:])
print("x[x > 5] =", x[x > 5])

## 9. 改变形状：reshape、view、flatten、squeeze、unsqueeze

| 方法 | 作用 | 常用参数 | 人话例子 |
|---|---|---|---|
| `reshape(shape(目标形状))` | 改成指定形状 | `shape(目标形状)` | `[12] -> [3, 4]` |
| `view(shape(目标形状))` | 改成指定形状，但要求内存连续 | `shape(目标形状)` | 老代码常见 |
| `flatten(start_dim(起始维度)=0)` | 拉平成一维或从某一维开始拉平 | `start_dim(起始维度)`, `end_dim(结束维度)` | 图片进全连接层前常用 |
| `unsqueeze(dim(插入位置))` | 在指定位置增加一个维度 | `dim(插入位置)` | `[3] -> [1, 3]` |
| `squeeze(dim(删除位置)=None)` | 删除长度为 1 的维度 | `dim(删除位置)` | `[1, 3, 1] -> [3]` |

核心要求：改形状前后，元素总数必须一样。`12` 个数可以改成 `[3, 4]`，不能改成 `[5, 5]`。

### 9.1 `unsqueeze(dim)` 的 dim 怎么理解

`unsqueeze(dim)` 里的 `dim` 不是“给元素插入下标”，而是**给形状插入一个新的维度位置**。

比如一个 Tensor 的形状是：

```python
[3, 2]
```

原来只有两个维度：

- `dim=0`：对应前面的 `3`
- `dim=1`：对应后面的 `2`

现在要插入一个新的长度为 1 的维度时，能插的位置有 3 个：最前面、中间、最后面。

可以把形状 `[3, 2]` 想成这样：

```text
_   3   _   2   _
0       1       2
```

这里的 `0, 1, 2` 是可以插入新维度的位置：

| 写法 | 插入位置 | 结果形状 |
|---|---|---|
| `x.unsqueeze(0)` | 插在最前面 | `[1, 3, 2]` |
| `x.unsqueeze(1)` | 插在 `3` 和 `2` 中间 | `[3, 1, 2]` |
| `x.unsqueeze(2)` | 插在最后面 | `[3, 2, 1]` |

一句话：`unsqueeze` 是在 shape 里插入一个 `1`。

In [ ]:
import torch

x = torch.arange(12)

print("original:", x, x.shape)
print("reshape(3, 4):\n", x.reshape(3, 4))  # shape(目标形状)
print("flatten:", x.reshape(3, 4).flatten())  # start_dim(起始维度), end_dim(结束维度)

v = torch.tensor([1, 2, 3])
print("v.shape:", v.shape)
print("v.unsqueeze(0).shape:", v.unsqueeze(0).shape)  # dim(插入位置)
print("v.unsqueeze(1).shape:", v.unsqueeze(1).shape)  # dim(插入位置)

y = torch.zeros(1, 3, 1)
print("y.shape:", y.shape)
print("y.squeeze().shape:", y.squeeze().shape)

### 9.2 用 `[3, 2]` 验证 `unsqueeze`

下面这段代码专门验证上面的插槽位理解。重点看 shape，不要只看 Tensor 里的数。

In [ ]:
import torch

x = torch.arange(6).reshape(3, 2)

print("x =\n", x)
print("原始形状 x.shape:", x.shape)

x0 = x.unsqueeze(0)
x1 = x.unsqueeze(1)
x2 = x.unsqueeze(2)

print("x.unsqueeze(0).shape:", x0.shape)  # [1, 3, 2]
print("x.unsqueeze(1).shape:", x1.shape)  # [3, 1, 2]
print("x.unsqueeze(2).shape:", x2.shape)  # [3, 2, 1]

print("\nx.unsqueeze(0) =\n", x0)
print("\nx.unsqueeze(1) =\n", x1)
print("\nx.unsqueeze(2) =\n", x2)

### 9.3 `reshape` 和 `view` 的区别

初学阶段先记一句：优先用 `reshape`，它更省心。

- `view` 要求 Tensor 的内存是连续的。
- `reshape` 会尽量返回 view；如果不行，可能复制一份新数据。
- 如果你明确想复制，用 `clone()`。
- 如果你遇到 `view` 报错，可以先试 `reshape` 或 `contiguous().view(...)`。

In [ ]:
x = torch.arange(12).reshape(3, 4)
x_t = x.t()  # 转置后通常不是连续内存

print("x_t.shape:", x_t.shape)
print("is_contiguous:", x_t.is_contiguous())
print("reshape works:", x_t.reshape(12))

# 如果一定要用 view，先 contiguous。
print("contiguous + view works:", x_t.contiguous().view(12))

## 10. 维度交换：transpose、t、permute

| 方法 | 作用 | 适合场景 |
|---|---|---|
| `x.t()` | 只适合二维矩阵转置 | 矩阵行列互换 |
| `x.transpose(dim0(维度0), dim1(维度1))` | 交换两个维度 | 任意维 Tensor 交换两个轴 |
| `x.permute(*dims(维度序列))` | 重新排列所有维度 | 图像通道顺序转换常用 |

### 10.1 `transpose(dim0, dim1)` 怎么理解

`transpose` 的意思是：**交换两个维度的位置**。

函数形式：

```python
x.transpose(dim0, dim1)
```

参数解释：

| 参数 | 含义 | 人话解释 |
|---|---|---|
| `dim0` | 第一个要交换的维度编号 | 你想拿哪个维度出来交换 |
| `dim1` | 第二个要交换的维度编号 | 和 `dim0` 互换位置的维度 |

比如一个 Tensor 的形状是：

```python
[2, 3, 4]
```

三个维度编号分别是：

```text
dim=0  dim=1  dim=2
  2      3      4
```

如果写：

```python
x.transpose(0, 1)
```

就是交换第 0 维和第 1 维，形状从 `[2, 3, 4]` 变成 `[3, 2, 4]`。

如果写：

```python
x.transpose(1, 2)
```

就是交换第 1 维和第 2 维，形状从 `[2, 3, 4]` 变成 `[2, 4, 3]`。

注意：`transpose` 不会增删元素，只是换维度顺序。

### 10.2 `t()`、`transpose()`、`permute()` 的区别

| 方法 | 能处理几维 | 做什么 |
|---|---:|---|
| `x.t()` | 只能二维 | 矩阵转置，等价于 `x.transpose(0, 1)` |
| `x.transpose(dim0, dim1)` | 任意维 | 只交换两个维度 |
| `x.permute(dims)` | 任意维 | 一次性重新排列所有维度 |

### 10.3 `permute(*dims)` 怎么理解

`permute` 的意思是：**按照你指定的新顺序，重新排列所有维度**。

函数形式：

```python
x.permute(*dims)
```

参数解释：

| 参数 | 含义 | 人话解释 |
|---|---|---|
| `*dims` | 新的维度排列顺序 | 把原来的维度编号按新顺序写一遍 |

注意：`permute` 里的参数不是目标 shape，而是**原维度编号的新顺序**。

比如一个 Tensor 原来的形状是：

```python
[2, 3, 4]
```

原维度编号是：

```text
dim=0  dim=1  dim=2
  2      3      4
```

如果写：

```python
x.permute(2, 0, 1)
```

意思是：

- 新的第 0 维，用原来的 `dim=2`，所以是 `4`
- 新的第 1 维，用原来的 `dim=0`，所以是 `2`
- 新的第 2 维，用原来的 `dim=1`，所以是 `3`

所以形状会从：

```text
[2, 3, 4]
```

变成：

```text
[4, 2, 3]
```

再比如图片数据：

| 格式 | shape | 含义 |
|---|---|---|
| `[H, W, C]` | `[高度, 宽度, 通道数]` | NumPy、Matplotlib 常见 |
| `[C, H, W]` | `[通道数, 高度, 宽度]` | PyTorch CNN 常见 |

如果图片是 `[224, 224, 3]`，要变成 PyTorch 常用的 `[3, 224, 224]`，就写：

```python
image.permute(2, 0, 1)
```

意思是：把原来的通道维 `dim=2` 放到最前面。

In [ ]:
import torch

# 二维矩阵：t() 等价于 transpose(0, 1)
matrix = torch.arange(6).reshape(2, 3)
print("matrix:\n", matrix)
print("matrix.shape:", matrix.shape)
print("matrix.t():\n", matrix.t())
print("matrix.t().shape:", matrix.t().shape)
print("matrix.transpose(0, 1):\n", matrix.transpose(0, 1))

print("-" * 40)

# 三维张量：transpose 每次只交换两个维度
x = torch.zeros(2, 3, 4)
print("x.shape:", x.shape)
print("x.transpose(0, 1).shape:", x.transpose(0, 1).shape)  # [3, 2, 4]
print("x.transpose(1, 2).shape:", x.transpose(1, 2).shape)  # [2, 4, 3]
print("x.permute(2, 0, 1).shape:", x.permute(2, 0, 1).shape)  # [4, 2, 3]

print("-" * 40)

image_hwc = torch.zeros(224, 224, 3)
image_chw = image_hwc.permute(2, 0, 1)
print("HWC:", image_hwc.shape)
print("CHW:", image_chw.shape)
print("image_hwc.permute(2, 0, 1).shape:", image_chw.shape)

### 10.4 拼接张量：`torch.cat` 和 `torch.stack`

拼接张量就是把多个 Tensor 合成一个更大的 Tensor。

PyTorch 里最常用的是两个函数：

| 方法 | 人话解释 | 是否新增维度 |
|---|---|---:|
| `torch.cat(tensors, dim=0)` | 沿着已有维度直接接起来 | 不新增维度 |
| `torch.stack(tensors, dim=0)` | 先创建一个新维度，再把张量叠起来 | 新增一个维度 |

### 10.4.1 `torch.cat(tensors, dim=0)`

函数形式：

```python
torch.cat(tensors, dim=0)
```

参数解释：

| 参数 | 含义 | 人话解释 |
|---|---|---|
| `tensors` | 要拼接的张量序列 | 通常写成列表或元组，比如 `[a, b]` |
| `dim` | 沿哪个已有维度拼接 | `dim=0` 沿行方向拼，`dim=1` 沿列方向拼 |

假设：

```python
a.shape = [2, 3]
b.shape = [2, 3]
```

- `torch.cat([a, b], dim=0)`：沿第 0 维拼，也就是上下拼，结果形状是 `[4, 3]`。
- `torch.cat([a, b], dim=1)`：沿第 1 维拼，也就是左右拼，结果形状是 `[2, 6]`。

注意：`cat` 是沿已有维度拼接，所以不会凭空多出一个新维度。

形状要求：除了拼接的那个 `dim` 可以不同，其他维度必须相同。

In [ ]:
import torch

a = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])

b = torch.tensor([
    [10, 20, 30],
    [40, 50, 60]
])

print("a.shape:", a.shape)
print("b.shape:", b.shape)

cat_dim0 = torch.cat([a, b], dim=0)
cat_dim1 = torch.cat([a, b], dim=1)

print("\ntorch.cat([a, b], dim=0)：上下拼")
print(cat_dim0)
print("shape:", cat_dim0.shape)

print("\ntorch.cat([a, b], dim=1)：左右拼")
print(cat_dim1)
print("shape:", cat_dim1.shape)

### 10.4.2 `torch.stack(tensors, dim=0)`

`stack` 和 `cat` 最大的区别：`stack` 会新增一个维度。

函数形式：

```python
torch.stack(tensors, dim=0)
```

参数解释：

| 参数 | 含义 | 人话解释 |
|---|---|---|
| `tensors` | 要堆叠的张量序列 | 这些 Tensor 的形状必须完全一样 |
| `dim` | 新维度插入的位置 | 和 `unsqueeze(dim)` 的插槽位理解类似 |

假设：

```python
v1.shape = [3]
v2.shape = [3]
```

- `torch.stack([v1, v2], dim=0)`：在最前面新增维度，结果形状是 `[2, 3]`。
- `torch.stack([v1, v2], dim=1)`：在中间新增维度，结果形状是 `[3, 2]`。

可以这样记：

```text
cat   ：沿旧维度拼，不新增维度
stack ：先新增一个维度，再沿新维度堆起来
```

In [ ]:
import torch

v1 = torch.tensor([1, 2, 3])
v2 = torch.tensor([10, 20, 30])

print("v1.shape:", v1.shape)
print("v2.shape:", v2.shape)

stack_dim0 = torch.stack([v1, v2], dim=0)
stack_dim1 = torch.stack([v1, v2], dim=1)

print("\ntorch.stack([v1, v2], dim=0)：")
print(stack_dim0)
print("shape:", stack_dim0.shape)

print("\ntorch.stack([v1, v2], dim=1)：")
print(stack_dim1)
print("shape:", stack_dim1.shape)

### 10.4.3 `cat` 和 `stack` 怎么选

| 你想做什么 | 用哪个 | 例子 |
|---|---|---|
| 把两批样本合成一批 | `cat` | `[32, 10] + [32, 10] -> [64, 10]` |
| 把两个特征矩阵左右拼起来 | `cat` | `[100, 3] + [100, 2] -> [100, 5]` |
| 把多个单独样本堆成 batch | `stack` | 10 个 `[3, 224, 224]` 图片 -> `[10, 3, 224, 224]` |
| 给多个结果新增一层组织维度 | `stack` | 多个 `[3]` 向量 -> `[数量, 3]` |

常见错误：

- `cat`：除了拼接的那个维度，其他维度必须一样。
- `stack`：所有输入 Tensor 的形状必须完全一样。
- `dim` 写错时，代码可能不报错，但 shape 会不是你想要的，所以拼接后一定打印 `.shape`。

## 11. 广播机制 broadcasting

广播就是：两个形状不同的 Tensor，在规则允许时，PyTorch 自动把小的那个“扩展”成能计算的形状。

最常见例子：一批样本加同一个偏置。

`x.shape = [3, 4]`，`bias.shape = [4]`，相加时 `bias` 会被当成 `[1, 4]`，再广播到 `[3, 4]`。

人话提醒：广播很方便，但也容易让 shape 错误不报错，所以每次写完关键计算都打印一下 shape。

In [ ]:
x = torch.ones(3, 4)
bias = torch.tensor([10, 20, 30, 40])

print("x.shape:", x.shape)
print("bias.shape:", bias.shape)
print(x + bias)